In [25]:
%load_ext autoreload
%autoreload 2

import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
from IPython.display import HTML, display
import os
from acpype.topol import ACTopol, MolTopol
import param_tool as pt
import py3Dmol
import MDAnalysis as mda
import nglview as nv
from parmed import amber, gromacs
import json


IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = (600, 400)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Парсер aminoacids.stp файла силовго поля
Мы парсим файл остатков силового поля, для того чтобы получить информацию какому атому в остатке соответствует какой тип атома 

In [2]:
import re

def parse_rtp_file(filepath):
    residues = {}  # Словарь для хранения информации об остатках
    current_residue = None  # Текущий остаток, который обрабатывается
    current_section = None  # Текущая секция (atoms, bonds, etc.)
    start_processing = False  # Флаг для начала обработки данных
    
    with open(filepath, 'r') as file:
        for line in file:
            line = line.strip()
            
            if '; Next are non-terminal AA\'s' in line:
                # Включаем режим сбора данных
                start_processing = True
                
            if not start_processing:
                continue  # Пропускаем строки до начала сбора данных
            
            # Проверяем, является ли строка названием остатка в квадратных скобках
            residue = re.findall(r'\[\s+([A-Z]{3,4})\s+\]', line)
            # Начало нового остатка
            if residue:
                # print(current_residue)
                current_residue = {
                'name': residue[0],  
                'atoms': {},
                'bonds': [],
                # 'valence': {},
                'impropers': []
                }
                residues[residue[0]] = current_residue
                
                current_section = None
                continue
            
            # Определение текущей секции
            section = re.findall(r'\[\s+([a-z]{5,})\s+\]', line)
            if section:
                section_name = line.strip('[]').strip()
                current_section = section_name
                continue
            
            # Сбор данных в зависимости от текущей секции
            if current_residue is not None:
                if current_section == 'atoms':
                    # Информация об атомах
                    parts = line.split()
                    current_residue['atoms'][parts[0]] = parts[1]
                        # 'name': ,
                        # 'type': parts[1],
                        # 'charge': float(parts[2]),
                        # 'index': int(parts[3])
                    # })
                elif current_section == 'bonds':
                    # Информация о связях
                    parts = line.split()
                    # print(parts)
                    current_residue['bonds'].append(tuple(parts))
                    # a1, a2 = parts
                    # print(a1)
                    # current_residue['valence'][] = 
                elif current_section == 'impropers':
                    if line:
                        # Неправильные связи
                        parts = line.split()
                        current_residue['impropers'].append(tuple(parts))
                else:
                    # Прочие секции
                    print(f"Unknown section: {current_section}", line)
    return residues

def save_charges_json(name: str, charge_list: list, path: str = ".") -> None:
    """
    Сохраняет список зарядов в JSON-файл с указанием абсолютного пути
    
    Параметры:
    name (str): Название файла (без расширения)
    charge_list (list): Список зарядов для сохранения
    path (str): Путь для сохранения (по умолчанию текущая директория)
    """
    # Создаем директорию, если она не существует
    os.makedirs(path, exist_ok=True)
    
    # Формируем полный путь к файлу
    file_path = os.path.join(path, f"{name}.json")
    absolute_path = os.path.abspath(file_path)
    
    # Сохраняем данные
    with open(file_path, 'w') as convert_file:
        json.dump(charge_list, convert_file, indent=4)
    
    # Выводим информативное сообщение
    print(f"Файл успешно сохранен: \n{absolute_path}")

In [3]:
!pwd

/home/_shared/_projects/2022_md_FRET_nv/param_R_CIT


In [4]:
# Пример использования
# filepath = "aminoacids.rtp"
# residues = parse_rtp_file(filepath)

In [5]:
# save_charges_json('aminoacids_dict', residues)

Файл успешно сохранен: 
/home/_shared/_projects/2022_md_FRET_nv/param_R_CIT/aminoacids_dict.json


In [6]:
residues['LYS']

{'name': 'LYS',
 'atoms': {'N': 'N',
  'H': 'H',
  'CA': 'CX',
  'HA': 'H1',
  'CB': 'C8',
  'HB1': 'HC',
  'HB2': 'HC',
  'CG': 'C8',
  'HG1': 'HC',
  'HG2': 'HC',
  'CD': 'C8',
  'HD1': 'HC',
  'HD2': 'HC',
  'CE': 'C8',
  'HE1': 'HP',
  'HE2': 'HP',
  'NZ': 'N3',
  'HZ1': 'H',
  'HZ2': 'H',
  'HZ3': 'H',
  'C': 'C',
  'O': 'O'},
 'bonds': [('N', 'H'),
  ('N', 'CA'),
  ('CA', 'HA'),
  ('CA', 'CB'),
  ('CA', 'C'),
  ('CB', 'HB1'),
  ('CB', 'HB2'),
  ('CB', 'CG'),
  ('CG', 'HG1'),
  ('CG', 'HG2'),
  ('CG', 'CD'),
  ('CD', 'HD1'),
  ('CD', 'HD2'),
  ('CD', 'CE'),
  ('CE', 'HE1'),
  ('CE', 'HE2'),
  ('CE', 'NZ'),
  ('NZ', 'HZ1'),
  ('NZ', 'HZ2'),
  ('NZ', 'HZ3'),
  ('C', 'O'),
  ('-C', 'N')],
 'impropers': [('-C', 'CA', 'N', 'H'), ('CA', '+N', 'C', 'O')]}

# Парсинг и модифицирование itp

In [2]:
def remove_extra_H(top, extra_pattern_H = 'HW'):
    atom_to_remove = [atom for atom in top.atoms if extra_pattern_H in atom.name]
    print(f"Найденные атомы для удаления: {atom_to_remove}")
    # Удаление атомов из списка атомов
    for atom in atom_to_remove[::-1]:
        top.atoms.remove(atom)

    # Ручное удаление связей, углов и диэдральных углов
    bonds_to_remove = [
        bond for bond in top.bonds if any(atom in atom_to_remove for atom in (bond.atom1, bond.atom2))
    ]
    angles_to_remove = [
        angle for angle in top.angles if any(atom in atom_to_remove for atom in (angle.atom1, angle.atom2, angle.atom3))
    ]
    dihedrals_to_remove = [
        dihedral for dihedral in top.dihedrals if any(atom in atom_to_remove for atom in (dihedral.atom1, dihedral.atom2, dihedral.atom3, dihedral.atom4))
    ]

    # Удаление связей
    for bond in bonds_to_remove:
        top.bonds.remove(bond)

    # Удаление углов
    for angle in angles_to_remove:
        top.angles.remove(angle)

    # Удаление диэдральных углов
    for dihedral in dihedrals_to_remove:
        top.dihedrals.remove(dihedral)

# Сохранение измененного файла топологии
# top.write(f"Acpype_data/{acpype_name}.acpype/{acpype_name}_GMX_cleaned.itp")

In [3]:
def check_atomtypes(top, path_to_atp = '', param_folder = ''):
    exist_types = []
    with open(path_to_atp, 'r') as atomtypes:
        file = atomtypes.readlines()
    for line in file:
        exist_types.append(line.split()[0])

    atom_types = {str(atom.atom_type) : atom.mass for atom in top}
    add_atom_types = []       
    for atom_type, atom_mass in atom_types.items():
        if atom_type not in exist_types:
            add_atom_types.append('%-2s%24.5f\n' % (atom_type, atom_mass))
    
    if param_folder:
        os.makedirs(param_folder, exist_ok=True)
    save_path = f'{param_folder}/atomtypes.atp'
    
    if add_atom_types:
        add_atom_types.extend(file)
        # print(f'In {path_to_atp}\nAdd line(-s):\n{add_atom_types}')
        
        with open (save_path, 'w') as atomtypes:
            atomtypes.writelines(add_atom_types)
        print(f'Добавлены недостающие типы атомов в {save_path}')
    else:
        with open (save_path, 'w') as atomtypes:
            atomtypes.writelines(file)
        print(f'Все используемые типы атомов указаны в {path_to_atp}')


In [4]:
def make_r2b(path_to_r2b = '', reference_aa = '', add_aa_name = '', param_folder = '', out=False):
    with open (path_to_r2b, 'r') as r2b:
        r2b_list = r2b.readlines()
    for i, r2b_line in enumerate(r2b_list):
        rtp_str = r2b_line.upper()
        if reference_aa.upper() == rtp_str.split()[0]:
            print(f'Replese old srt:\n{rtp_str}')
            add_rtp_line = rtp_str.replace('-', add_aa_name.upper(),1)
            print(f'To new str:\n{add_rtp_line}')
            r2b_list[i] = add_rtp_line

    save_path = f'{param_folder}/aminoacids.r2b'
    with open(save_path, 'w') as f:
        f.writelines(r2b_list)
    print(f'Save in  {save_path}')

## Уделение атомов не входящих в структуру остатка

In [90]:
!pwd

/home/_shared/_projects/2022_md_FRET_nv/param_R_CIT


In [26]:
PTM_name = 'Lysine_Butyryl'
acpype_name = 'Lysine_Butyryl_rn_H_3D'
base_name = 'KBU'

In [27]:
top_L = gromacs.GromacsTopologyFile(f"{PTM_name}/Acpype_data/{acpype_name}.acpype/{acpype_name}_GMX.itp")
top_L.atoms

AtomList([
	<Atom N [0]; In KBU 0>
	<Atom CA [1]; In KBU 0>
	<Atom C [2]; In KBU 0>
	<Atom O [3]; In KBU 0>
	<Atom CB [4]; In KBU 0>
	<Atom CG [5]; In KBU 0>
	<Atom CD [6]; In KBU 0>
	<Atom CE [7]; In KBU 0>
	<Atom NZ [8]; In KBU 0>
	<Atom CH [9]; In KBU 0>
	<Atom OT2 [10]; In KBU 0>
	<Atom CT1 [11]; In KBU 0>
	<Atom CI [12]; In KBU 0>
	<Atom CK [13]; In KBU 0>
	<Atom HK3 [14]; In KBU 0>
	<Atom HK2 [15]; In KBU 0>
	<Atom HK1 [16]; In KBU 0>
	<Atom HI2 [17]; In KBU 0>
	<Atom HI1 [18]; In KBU 0>
	<Atom HT12 [19]; In KBU 0>
	<Atom HT11 [20]; In KBU 0>
	<Atom HZ [21]; In KBU 0>
	<Atom HE2 [22]; In KBU 0>
	<Atom HE1 [23]; In KBU 0>
	...
	<Atom HB1 [29]; In KBU 0>
	<Atom HA [30]; In KBU 0>
	<Atom H [31]; In KBU 0>
	<Atom HW1 [32]; In KBU 0>
	<Atom HW2 [33]; In KBU 0>
])

In [28]:
remove_extra_H(top_L, extra_pattern_H = 'W')

Найденные атомы для удаления: [<Atom HW1 [32]; In KBU 0>, <Atom HW2 [33]; In KBU 0>]


## Меняем типа атомов в составе itp на те что прописаны для атомов в стандартном остатке 

In [29]:
!pwd

/home/_shared/_projects/2022_md_FRET_nv/param_R_CIT


In [30]:
# Загружаем распарсенный aminoacids rtp c информацией о параметризованных остатках
with open ('aminoacids_dict.json', 'r') as f:
    residues = json.load(f)
# Загружаем имена атомов что должны наследовать тип атомов из оригинальной аминокислоты
with open (f'{PTM_name}/general_atoms.json', 'r') as f:
    general_atoms = json.load(f)
general_atoms

['H',
 'N',
 'CA',
 'HA',
 'CB',
 'HB2',
 'HB1',
 'CG',
 'HG2',
 'HG1',
 'CD',
 'HD2',
 'HD1',
 'CE',
 'HE2',
 'HE1',
 'C',
 'O']

In [31]:
# Присваиваем типы атомов из оригинальной а.к.
# Важно: атом может быть каноническим, но его валентность могла поменяться, следи за этим! 
# (следим на шаге сопоставления и переименовки атомов)
aminoacid = 'LYS'
exclusion_atoms = ['HE1', 'HE2']
canonical_atoms = [name for name in general_atoms if name not in exclusion_atoms] 

print('Для общих атомов между каноничным остатком и модифицированным типы атомов меняются на каноничные',
      'Col 1: Имя атома в параметризуемом остатке',
      'Col 2: Тип атома из itp параметризуемого остатка',
      'Col 3: Тип атома из rtp каноничного остатка',
      'Col 4: Итоговый тип атома в будующем rtp',
      'Name | Atom Type from mod_acid (itp) | Сanonical Atom Type (rtp) | Final atom type (new rtp)', sep ='\n')
for atom in top_L:
    if atom.name in general_atoms: # общие имена только у эквивалентных атомов  
        atype = residues[aminoacid]['atoms'][atom.name]
        if atom.atom_type != atype and atom.name not in exclusion_atoms:
            print("%4s%5s%5s%5s" % (atom.name, atom.atom_type, ("%5s" % atype).replace(atype, "\033[48;5;203m%s\033[0m"  % atype), atype))
            atom.atom_type = atype
        else:
            atom_type = str(atom.atom_type)
            print("%4s%5s%5s%5s" % (atom.name, ("%5s" % atom_type).replace(atom_type.strip(), "\033[48;5;71m%s\033[0m" % atom_type.strip()), atype, atom_type))
    else:
        at = atom.atom_type.name
        print("%4s%5s%5s%5s" % (atom.name, ("%5s" % at).replace(at.strip(), "\033[48;5;71m%s\033[0m" % at), '-', at ))
        
#         \033[48;5;203m%s\033[0m "\033[48;5;203m%s\033[0m"

Для общих атомов между каноничным остатком и модифицированным типы атомов меняются на каноничные
Col 1: Имя атома в параметризуемом остатке
Col 2: Тип атома из itp параметризуемого остатка
Col 3: Тип атома из rtp каноничного остатка
Col 4: Итоговый тип атома в будующем rtp
Name | Atom Type from mod_acid (itp) | Сanonical Atom Type (rtp) | Final atom type (new rtp)
   N   NT    N    N
  CA   CT   CX   CX
   C    C    C    C
   O    O    O    O
  CB   CT   C8   C8
  CG   CT   C8   C8
  CD   CT   C8   C8
  CE   CT   C8   C8
  NZ    N    -    N
  CH    C    -    C
 OT2    O    -    O
 CT1   CT    -   CT
  CI   CT    -   CT
  CK   CT    -   CT
 HK3   HC    -   HC
 HK2   HC    -   HC
 HK1   HC    -   HC
 HI2   HC    -   HC
 HI1   HC    -   HC
HT12   HC    -   HC
HT11   HC    -   HC
  HZ    H    -    H
 HE2   H1   HP   H1
 HE1   H1   HP   H1
 HD2   HC   HC   HC
 HD1   HC   HC   HC
 HG2   HC   HC   HC
 HG1   HC   HC   HC
 HB2   HC   HC   HC
 HB1   HC   HC   HC
  HA   H1   H1   H1
   H    H    

In [32]:
residues[aminoacid]['atoms']

{'N': 'N',
 'H': 'H',
 'CA': 'CX',
 'HA': 'H1',
 'CB': 'C8',
 'HB1': 'HC',
 'HB2': 'HC',
 'CG': 'C8',
 'HG1': 'HC',
 'HG2': 'HC',
 'CD': 'C8',
 'HD1': 'HC',
 'HD2': 'HC',
 'CE': 'C8',
 'HE1': 'HP',
 'HE2': 'HP',
 'NZ': 'N3',
 'HZ1': 'H',
 'HZ2': 'H',
 'HZ3': 'H',
 'C': 'C',
 'O': 'O'}

In [33]:
for atom in top_L:
    if atom.name in residues['LYS']['atoms'].keys():
        print(atom.name,residues['LYS']['atoms'][atom.name], atom.atom_type)


N N N
CA CX CX
C C C
O O O
CB C8 C8
CG C8 C8
CD C8 C8
CE C8 C8
NZ N3 N
HE2 HP H1
HE1 HP H1
HD2 HC HC
HD1 HC HC
HG2 HC HC
HG1 HC HC
HB2 HC HC
HB1 HC HC
HA H1 H1
H H H


### Генерируем apt файл с типами атомов 

In [34]:
# check_atomtypes(top_L, path_to_atp = 'amber14sb_parmbsc1_cufix.ff/atomtypes.atp', param_folder = f'{PTM_name}/force_field_files')
check_atomtypes(top_L, path_to_atp = 'amber14sb_mod.ff/atomtypes.atp', param_folder = f'amber14sb_mod.ff/')

Все используемые типы атомов указаны в amber14sb_mod.ff/atomtypes.atp


## Создание rtp файла 

In [35]:
with open(f'{PTM_name}/partial_charges.json', 'r') as charges:
    mod_acid_charges = json.load(charges)
print(f'{round(sum(mod_acid_charges), 4):.4}',mod_acid_charges, sep = '\n')

-0.0
[-0.3479, -0.24, 0.7341, -0.5894, -0.0094, 0.0187, -0.0479, -0.0436, -0.6245, 0.6603, -0.578, -0.3636, 0.0566, -0.4504, 0.1396, 0.1396, 0.1396, 0.0416, 0.0416, 0.1005, 0.1005, 0.3091, 0.0892, 0.0892, 0.0621, 0.0621, 0.0103, 0.0103, 0.0362, 0.0362, 0.1426, 0.2747]


In [36]:
def make_rtp_for_aminoacid(g_parm,charges, name, c5=[],c3=[],shift = 0,canonical_atoms = [], add_inter_param = False):
    rtp_text = ['[ bondedtypes ]\n',
                '; Col 1: Type of bond\n',
                '; Col 2: Type of angles\n',
                '; Col 3: Type of proper dihedrals\n',
                '; Col 4: Type of improper dihedrals\n',
                '; Col 5: Generate all dihedrals if 1, only heavy atoms of 0.\n',
                '; Col 6: Number of excluded neighbors for nonbonded interactions\n',
                '; Col 7: Generate 1,4 interactions between pairs of hydrogens if 1\n',
                '; Col 8: Remove impropers over the same bond as a proper if it is 1\n',
                '; bonds  angles  dihedrals  impropers all_dihedrals nrexcl HH14 RemoveDih\n',
                '     1       1          9          4        1         3      1     0 \n', '\n']
    
    rtp_text.append(f'[ {name} ]\n')
    
    rtp_text.append(' [ atoms ]\n')
    for i, atom in enumerate(g_parm.atoms):
        rtp_text.append("%6s%6s%12.4f%4d\n"% (atom.name, str(atom.atom_type), charges[i], atom.idx +shift))
    rtp_text.append(' [ bonds ]\n')
    for bond in g_parm.bonds:
        if (bond.atom1.name and bond.atom2.name) in canonical_atoms:
            rtp_text.append("%6s%6s\n" % (bond.atom1.name, bond.atom2.name))
        else: 
            rtp_text.append("%6s%6s%12.4f%12.1f\n" % (bond.atom1.name, bond.atom2.name, 0.1*bond.type.req, bond.type.k*2*4.184*100))
    rtp_text.append('   -C    N\n')
    rtp_text.append(' [ angles ]\n')
    for angle in g_parm.angles:
        if (angle.atom1.name and angle.atom2.name and angle.atom3.name) in canonical_atoms:
            rtp_text.append("%6s%6s%6s\n" % (angle.atom1.name, angle.atom2.name, angle.atom3.name))
        else: 
            rtp_text.append("%6s%6s%6s %12.4f %8.1f\n" % (angle.atom1.name, angle.atom2.name, angle.atom3.name, 
                                              angle.type.theteq, angle.type.k*2*4.184))
    rtp_text.append(' [ dihedrals ]\n')
    for dihedral in g_parm.dihedrals:
        if dihedral.funct == 9 and (dihedral.atom1.name and 
                                    dihedral.atom2.name and 
                                    dihedral.atom3.name and 
                                    dihedral.atom4.name) in canonical_atoms:
            
            rtp_text.append("%6s%6s%6s%6s\n" % (dihedral.atom1.name, dihedral.atom2.name,
                                            dihedral.atom3.name, dihedral.atom4.name))
        elif dihedral.funct == 9:
            for i in dihedral.type:
                rtp_text.append("%6s%6s%6s%6s %8.3f %8.4f %5i\n" % (dihedral.atom1.name, dihedral.atom2.name,
                                                                       dihedral.atom3.name, dihedral.atom4.name,
                                                                       i.phase, i.phi_k*4.184,
                                                                       i.per))
    rtp_text.append(' [ impropers ]\n')
    for dihedral in g_parm.dihedrals:
        if dihedral.funct == 4 and (dihedral.atom1.name and 
                                    dihedral.atom2.name and 
                                    dihedral.atom3.name and 
                                    dihedral.atom4.name) in canonical_atoms:
            rtp_text.append("%6s%6s%6s%6s\n" % (dihedral.atom1.name, dihedral.atom2.name,
                                                dihedral.atom3.name, dihedral.atom4.name))
        elif dihedral.funct == 4:
            # for i in dihedral.type:
            rtp_text.append("%6s%6s%6s%6s %8.3f %8.4f %5i\n" % (dihedral.atom1.name, dihedral.atom2.name,
                                                               dihedral.atom3.name, dihedral.atom4.name,
                                                               dihedral.type.phase, dihedral.type.phi_k*4.184,
                                                               dihedral.type.per))
            
    rtp_text.extend(['    -C    CA     N     H\n',
                     '    CA    +N     C     O\n'])
    out = ''.join(rtp_text)
    
    return out

In [37]:

name = PTM_name
PTM_name

rtp = make_rtp_for_aminoacid(top_L, charges=mod_acid_charges, name=base_name, shift=1,canonical_atoms = canonical_atoms)
# print(rtp)
with open(f'{PTM_name}/force_field_files/{name}.rtp' , 'w') as f: 
    f.write(rtp)
with open(f'amber14sb_mod.ff/{name}.rtp' , 'w') as f: 
    f.write(rtp)

In [23]:
# print(rtp)

[ bondedtypes ]
; Col 1: Type of bond
; Col 2: Type of angles
; Col 3: Type of proper dihedrals
; Col 4: Type of improper dihedrals
; Col 5: Generate all dihedrals if 1, only heavy atoms of 0.
; Col 6: Number of excluded neighbors for nonbonded interactions
; Col 7: Generate 1,4 interactions between pairs of hydrogens if 1
; Col 8: Remove impropers over the same bond as a proper if it is 1
; bonds  angles  dihedrals  impropers all_dihedrals nrexcl HH14 RemoveDih
     1       1          9          4        1         3      1     0 

[ KAC ]
 [ atoms ]
     N     N     -0.4157   1
    CA    CX      0.0028   2
     C     C      0.5972   3
     O     O     -0.5679   4
    CB    C8     -0.2323   5
    CG    C8      0.1558   6
    CD    C8      0.0660   7
    CE    C8     -0.2130   8
    NZ     N     -0.7030   9
    CH     C      0.8572  10
   OT2     O     -0.6091  11
   CT1    CT     -0.4775  12
  HT13    HC      0.1241  13
  HT12    HC      0.1241  14
  HT11    HC      0.1241  15
    HZ  

## Создание файла r2b

In [100]:
# param_folder = f'{PTM_name}/force_field_files/'
# path_to_r2b = 'amber14sb_parmbsc1_cufix.ff/aminoacids.r2b'
# reference_aa = 'Lys'
# make_r2b(path_to_r2b, reference_aa, base_name, param_folder)

Replese old srt:
LYS    LYS   NLYS  CLYS  -

To new str:
LYS    LYS   NLYS  CLYS  KMA

Save in  Lysine_1M_ACC//aminoacids.r2b


## Созданиек dat файла 

In [38]:
path_to_dat = 'amber14sb_parmbsc1_cufix.ff/residuetypes.dat'


with open (path_to_dat, 'r') as dat:
    dat_list = dat.readlines()
dat_list = [f'{base_name}     Protein\n']+dat_list
save_path = f'{PTM_name}/force_field_files/residuetypes.dat'
with open (save_path, 'w') as dat:
    dat.writelines(dat_list)
print(f'In {save_path}\nAdd line:\n{base_name}     Protein')

In Lysine_Butyryl/force_field_files/residuetypes.dat
Add line:
KBU     Protein


In [39]:
path_to_dat = 'amber14sb_mod.ff/residuetypes.dat'


with open (path_to_dat, 'r') as dat:
    dat_list = dat.readlines()
dat_list = [f'{base_name}     Protein\n']+dat_list

save_path = f'amber14sb_mod.ff/residuetypes.dat'
with open (save_path, 'w') as dat:
    dat.writelines(dat_list)
print(f'In {save_path}\nAdd line:\n{base_name}     Protein')

In amber14sb_mod.ff/residuetypes.dat
Add line:
KBU     Protein


## Создание am файла 

In [40]:
path_to_am = 'amber14sb_parmbsc1_cufix.ff/Makefile.am'


edd_file = []
with open(path_to_am, 'r') as am:
    for line in am:
        if line.startswith('topol_DATA'):
            line = line.replace('\\', f'{PTM_name}_raw.rtp \\')
            # print(line.split())
        edd_file.append(line)
save_path = f'{PTM_name}/force_field_files/Makefile.am'
with open (save_path, 'w') as dat:
    dat.writelines(edd_file)
# print(f'In {save_path}\nAdd line:\n{add_aa_name}     Protein')

In [41]:
path_to_am = 'amber14sb_mod.ff/Makefile.am'


edd_file = []
with open(path_to_am, 'r') as am:
    for line in am:
        if line.startswith('topol_DATA'):
            line = line.replace('\\', f'{PTM_name}.rtp \\')
            # print(line.split())
        edd_file.append(line)
save_path = f'amber14sb_mod.ff/Makefile.am'
with open (save_path, 'w') as dat:
    dat.writelines(edd_file)
# print(f'In {save_path}\nAdd line:\n{add_aa_name}     Protein')

## Создание in файла 

In [26]:
path_to_am = 'amber14sb_parmbsc1_cufix.ff/Makefile.in'


edd_file = []
with open(path_to_am, 'r') as am:
    for line in am:
        if line.startswith('topol_DATA'):
            line = line.replace('\\', f'{PTM_name}_raw.rtp \\')
            # print(line.split())
        edd_file.append(line)
save_path = f'{PTM_name}/force_field_files/Makefile.in'
with open (save_path, 'w') as dat:
    dat.writelines(edd_file)

## Создание hdb файла

In [42]:
from rdkit import Chem
from collections import defaultdict

def hdb_generator(pdb_file, resname='MOD', resid=1, segid='A'):
    """
    Модифицирует имена атомов в молекуле по правилам аминокислот:
    - N, H, CA, C, O имеют стандартные имена
    - Боковые атомы получают буквенные метки по греческому алфавиту
    - Протоны наследуют имя родительского атома и получают числовой суффикс
    """
    def find_common_prefix(strings):
        if not strings:
            return ""

        first = strings[0]
        for i in range(len(first), 0, -1):
            prefix = first[:i]
            if all(s.startswith(prefix) for s in strings[1:]):
                return prefix
        return ""

    hdb = ['1	1	H	N	-C	CA	\n',
           '1	5	HA	CA	N	CB	C\n']
    rdkit_mol = pt.pdb_to_chem(pdb_file, removeHs=False)
    rdkit_mol = next(iter(rdkit_mol.values()))

    if rdkit_mol is None:
        raise ValueError("Не удалось загрузить молекулу из PDB-файла")

    # heavy_atoms = {}
    for atom in rdkit_mol.GetAtoms():
        if atom.GetSymbol() == 'H':
            continue
        current_idx = atom.GetIdx()
        current_atom_name = atom.GetProp('AtomName')
        
        protons = []
        visited_atoms = {current_atom_name: current_idx}
        for neighbor_atom in atom.GetNeighbors():
            atom_name = neighbor_atom.GetProp('AtomName')
            atom_idx = neighbor_atom.GetIdx()
            if neighbor_atom.GetAtomicNum() != 1:
                visited_atoms[atom_name] = atom_idx
            else:
                protons.append(atom_name)
                # neighbors[atom.GetAtomicNum()] += 1
        if len(visited_atoms) == 2 and 'H' not in protons: # Если Это концевой атом то надо добавить еще один
            atom_3_idx = min(visited_atoms.values())-1
            atom_name = rdkit_mol.GetAtomWithIdx(atom_3_idx).GetProp('AtomName')
            visited_atoms[atom_name] = atom_3_idx
        if atom.GetAtomicNum() == 6:
            if max(visited_atoms.values()) == current_idx:
                geom_n = 4
            elif len(protons) == 1 and len(visited_atoms) == 3: # sp2 углерод с 1 протоном
                geom_n = 1
            elif len(protons) == 1 and len(visited_atoms) == 4: # sp3 углерод с 1 протоном
                geom_n = 5
            else:
                geom_n = 6
        elif atom.GetAtomicNum() == 7:
            if max(visited_atoms.values()) == current_idx: # атом N последний
                geom_n = 3
            else:
                geom_n = 1
        
        print(protons)
        if protons: # ['H'] ['HB1', 'HB2'] ['HC1', 'HC2', 'HC3']
            n_H = str(len(protons))
            H_name = find_common_prefix(protons)
            if H_name not in ['H', 'HA']:
                hdb.append("{}\t{}\t{}\t{}\n".format(n_H, geom_n if geom_n else 'N', H_name, "\t".join(visited_atoms)))# ['CB', 'CA', 'CG'] ['CK', 'CI'] ['CA', 'N', 'C', 'CB']
    # print(heavy_atoms)
    hdb = [f'{resname}\t{len(hdb)}\n'] + hdb
    return hdb
    #     print(neighbors, sep = '\n')
    # print(heavy_atoms)



In [43]:
# Пример использования:
add_hdb = hdb_generator(f'{PTM_name}/molecules/substructure/{PTM_name}_rn_3D.pdb', base_name)
print(*add_hdb, sep = '\n')
edd_file = []


['H']
['HA']
[]
[]
['HB2', 'HB1']
['HG2', 'HG1']
['HD2', 'HD1']
['HE2', 'HE1']
['HZ']
[]
[]
['HT12', 'HT11']
['HI2', 'HI1']
['HK3', 'HK2', 'HK1']
KBU	10

1	1	H	N	-C	CA	

1	5	HA	CA	N	CB	C

2	6	HB	CB	CA	CG

2	6	HG	CG	CB	CD

2	6	HD	CD	CG	CE

2	6	HE	CE	CD	NZ

1	1	HZ	NZ	CE	CH

2	6	HT1	CT1	CH	CI

2	6	HI	CI	CT1	CK

3	4	HK	CK	CI	CT1



In [44]:
path_to_hdb = 'amber14sb_parmbsc1_cufix.ff/aminoacids.hdb'
hdb_file = []
with open(path_to_hdb, 'r') as hdb:
    for line in hdb:
        hdb_file.append(line)


save_path = f'{PTM_name}/force_field_files/aminoacids.hdb'
with open (save_path, 'w') as dat:
    dat.writelines(add_hdb+hdb_file)

In [45]:
path_to_hdb = 'amber14sb_mod.ff/aminoacids.hdb'
hdb_file = []
with open(path_to_hdb, 'r') as hdb:
    for line in hdb:
        hdb_file.append(line)


save_path = f'amber14sb_mod.ff/aminoacids.hdb'
with open (save_path, 'w') as dat:
    dat.writelines(add_hdb+hdb_file)